In [ ]:
""" Task 4: Heatwaves """

In [ ]:
import requests
import pandas as pd
import yaml

endpoint = 'https://frost.met.no/observations/v0.jsonld'
parameters = {
    'sources': 'SN17850',
    'elements': 'max(air_temperature P1D)',
    'referencetime': '2025-01-01/2026-01-01'
}

client_id = "unique-client-id"  #use your actual Frost API client_id here
response = requests.get(endpoint, params=parameters, auth=(client_id, ''))
print(f"API Request Status: {response.status_code}")

if response.status_code == 200:
    payload = response.json()
    
    data = [{
        "Time": pd.to_datetime(entry["referenceTime"]).strftime("%Y-%m-%d"),
        "Max Temp": entry["observations"][0]["value"]
    } for entry in payload.get("data", [])]
    
    df = pd.DataFrame(data)
    
    df['heat_day'] = df['Max Temp'] >= 27
    df['group'] = (df['heat_day'] != df['heat_day'].shift()).cumsum()
    
    heatwave_dates = []
    for _, group in df[df['heat_day']].groupby('group'):
        if len(group) >= 5:
            heatwave_dates.append(f"{group['Time'].iloc[0]} - {group['Time'].iloc[-1]}")
            
    summary_dict = {
        'dates': heatwave_dates,
        'station_id': 'SN17850',
        'station_name': 'Ås',
        'year': 2025
    }
    
    with open('heatwave_summary_SN17850_2025.yml', 'w', encoding='utf-8') as f:
        yaml.dump(summary_dict, f, sort_keys=False, allow_unicode=True)
        
    print("Heatwave summary successfully saved to YAML!")
    print(yaml.dump(summary_dict, sort_keys=False, allow_unicode=True))
else:
    print("Failed to fetch data. Check your client_id.")

API Request Status: 200
Heatwave summary successfully saved to YAML!
dates:
- 2025-07-16 - 2025-07-20
station_id: SN17850
station_name: Ås
year: 2025



From the results we can see that we had one period of five concecutive days with maximum temperature above 27 C.